In [0]:
%pip install kagglehub==1.0.1

In [0]:
import kagglehub
import os
import shutil
import glob
import json

# ============================================================
# CONFIGURAÇÃO — ajuste apenas estes valores
# ============================================================

CATALOG = "northwind_raw_data"
SCHEMA  = "source_kaggle_api"                # schema onde o volume está criado
VOLUME  = "vol_extracted_data"

KAGGLE_DATASET = "jeetahirwar/northwind-traders"

# Caminho do volume no Databricks (não altere a estrutura)
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# ============================================================
# GARANTE QUE O VOLUME EXISTE
# ============================================================

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

# ============================================================
# CONFIGURA CREDENCIAIS DA API KAGGLE
# ============================================================

# Obtém as credenciais dos secrets
kaggle_username = dbutils.secrets.get(scope="northwind", key="KAGGLE_USERNAME")
kaggle_key = dbutils.secrets.get(scope="northwind", key="API_TOKEN")

# Cria o diretório .kaggle no home do usuário
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

# Cria o arquivo kaggle.json com as credenciais
kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
kaggle_credentials = {
    "username": kaggle_username,
    "key": kaggle_key
}

with open(kaggle_json_path, "w") as f:
    json.dump(kaggle_credentials, f)

# Define as permissões corretas (necessário para a API Kaggle)
os.chmod(kaggle_json_path, 0o600)

# ============================================================
# DOWNLOAD DOS CSVs VIA KAGGLE HUB
# ============================================================

local_path = kagglehub.dataset_download(KAGGLE_DATASET)
print(f"Download concluído em: {local_path}")

# ============================================================
# MOVE OS CSVs PARA O VOLUME
# ============================================================

csv_files = glob.glob(os.path.join(local_path, "**", "*.csv"), recursive=True)

if not csv_files:
    raise FileNotFoundError(f"Nenhum arquivo CSV encontrado em: {local_path}")

print(f"\n{len(csv_files)} arquivo(s) encontrado(s). Copiando para o volume...")

for src in csv_files:
    filename = os.path.basename(src)
    dst = os.path.join(VOLUME_PATH, filename)
    shutil.copy2(src, dst)
    print(f"  ✓ {filename}")

print(f"\nTodos os arquivos estão em: {VOLUME_PATH}")

# ============================================================
# VALIDAÇÃO — lista o conteúdo do volume
# ============================================================

print("\nConteúdo do volume:")
for f in os.listdir(VOLUME_PATH):
    size_kb = os.path.getsize(os.path.join(VOLUME_PATH, f)) / 1024
    print(f"  {f:<40} {size_kb:>8.1f} KB")